## Executive Summary

**Purpose:** Check that the teeth of a planetary set survive the load for the required life: pitting of the flanks (contact stress) and fatigue breakage at the root (bending stress), for the sun–planet and planet–ring meshes.

**What it does:** `rate_planetary` splits the output torque over the planets, applies the load factors of ISO 6336-1, and computes the contact stress of ISO 6336-2 and the tooth-root stress of ISO 6336-3 for each gear, with the life factors for the number of load cycles each gear sees. It returns the safety factors against pitting $S_H$ and against root breakage $S_F$, and names the gear and failure mode that limit the design.

**Method:** ISO 6336:2019 Method B for the geometry factors, with Method C simplifications for the dynamic factor $K_v$, the face load factor $K_{H\beta}$ (entered, with a default) and the transverse load factors $K_{H\alpha}$, $K_{F\alpha}$. The tooth form factor uses load at the tip with the contact ratio factor $Y_\varepsilon$ (Method C of ISO 6336-3:1996), which is a little conservative compared with loading at the outer point of single tooth contact. This is a design calculation. A design that is close to its limits, or a safety-critical one, should be re-checked in KISSsoft, RIKOR or a full ISO 6336 tool, and the material values confirmed with the heat treater.

## How It Fits Into the Gear Design Workflow

**Upstream:** `gears.layout.PlanetarySet` (geometry and contact ratios of both meshes).

**Downstream:** `gears.components` (the tooth forces load the planet pins and bearings), `gears.design` (the search keeps sets whose safety factors meet the minima).

In [ ]:
#| default_exp gears.rating

In [ ]:
#| export
from __future__ import annotations

import math
from dataclasses import dataclass, field

from emachines.gears.involute import GearGeometry, inv
from emachines.gears.layout import PlanetarySet

In [ ]:
#| hide
# drawing helpers for the explanatory figures (not exported)
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Arc, Polygon, Wedge, Rectangle
from emachines.gears import layout as _lay, rating as _rat     # exported modules, so figures can sit ahead of the definitions
from emachines.gears.involute import GearGeometry as _GG
from emachines.gears.manufacturing import gear_outline

INK, BLUE, RED, GREEN = "#222222", "#1f5fa8", "#b8322a", "#2a8a4a"
C_SUN, C_PLANET, C_RING, C_STEEL = "#d8e8d0", "#f6e7c8", "#dfe7f1", "#bdbdbd"

def _rot(P, a):
    c, s_ = np.cos(a), np.sin(a)
    return np.asarray(P) @ np.array([[c, s_], [-s_, c]])

def _label(ax, text, xy, xytext, color=INK, ha="left", fs=9):
    ax.annotate(text, xy=xy, xytext=xytext, fontsize=fs, color=color, ha=ha, va="center",
                arrowprops=dict(arrowstyle="-", color=color, lw=0.7, shrinkA=0, shrinkB=0))

def _dim(ax, p0, p1, text="", off=(0, 0), color=BLUE, fs=9, ha="center"):
    ax.annotate("", xy=p1, xytext=p0, arrowprops=dict(arrowstyle="<->", color=color, lw=0.8, shrinkA=0, shrinkB=0))
    if text:
        ax.text((p0[0] + p1[0]) / 2 + off[0], (p0[1] + p1[1]) / 2 + off[1], text, color=color, fontsize=fs,
                ha=ha, va="center", bbox=dict(fc="white", ec="none", pad=0.6, alpha=0.85))

def _circ(ax, r, style="--", color=BLUE, lw=0.7, c=(0, 0)):
    ax.add_patch(Circle(c, r, fill=False, ls=style, ec=color, lw=lw))

def _set_drawing(ps, ax, turn=np.pi / 2):
    "Draw sun, planets and ring of a PlanetarySet in mesh, turned so planet 1 is at angle `turn`."
    rho = np.pi * ps.planet.z / ps.ring.z
    ax.add_patch(Polygon(_rot(gear_outline(ps.ring), rho + turn), closed=True, fc="white", ec=INK, lw=0.7))
    for k in range(ps.n_planets):
        th = 2 * np.pi * k / ps.n_planets
        psi = th * (1 + ps.sun.z / ps.planet.z) + np.pi - np.pi / ps.planet.z
        c = ps.a_w * np.array([np.cos(th), np.sin(th)])
        ax.add_patch(Polygon(_rot(_rot(gear_outline(ps.planet), psi) + c, turn), closed=True, fc=C_PLANET, ec=INK, lw=0.7))
    ax.add_patch(Polygon(_rot(gear_outline(ps.sun), turn), closed=True, fc=C_SUN, ec=INK, lw=0.7))

## Materials

Allowable stress numbers for the endurance limit, from ISO 6336-5 for quality grade MQ (the grade a good commercial heat treater achieves with process control). $\sigma_{H\,lim}$ is the contact endurance limit and $\sigma_{F\,lim}$ the bending endurance limit of the standard reference test gear; the bending strength of a single tooth loaded in one direction is $\sigma_{FE} = 2\sigma_{F\,lim}$.

| Key | Steel and treatment | $\sigma_{H\,lim}$ | $\sigma_{F\,lim}$ | Typical use |
|---|---|---|---|---|
| `case_carburized` | 20MnCr5, 18CrNiMo7-6; carburized, 58–62 HRC, core ≥ 30 HRC | 1500 | 500 | sun, planets |
| `nitrided` | 31CrMoV9 nitriding steel, gas nitrided, ≥ 650 HV surface | 1250 | 420 | ring, planets |
| `qt_nitrided` | 42CrMo4 quenched and tempered, then nitrided, ≥ 500 HV | 1000 | 360 | ring |
| `through_hardened` | 42CrMo4 quenched and tempered to about 300 HB (315 HV) | 787 | 321 | ring, low-cost sets |

The through-hardened values come from the ISO 6336-5 MQ lines $\sigma_{H\,lim} = 1.313\,HV + 373$ and $\sigma_{F\,lim} = 0.425\,HV + 187$. The others are the MQ plateau values. Treat all of them as typical; your supplier's certificate or the heat treater's own data takes precedence.

In [ ]:
#| export
@dataclass(frozen=True)
class GearMaterial:
    """
    Steel and heat treatment of one gear.

    name       : str    Designation for the drawing
    treatment  : str    Heat treatment for the drawing
    hardness   : str    Surface (and core) hardness for the drawing
    sigma_Hlim : float  Contact endurance limit [MPa]
    sigma_Flim : float  Bending endurance limit [MPa]
    kind       : str    'case' (carburized/induction), 'nitrided' or 'through' — selects the life curves
    E, nu      : float  Young's modulus [MPa] and Poisson's ratio
    """
    name: str
    treatment: str
    hardness: str
    sigma_Hlim: float
    sigma_Flim: float
    kind: str
    E: float = 206_000.0
    nu: float = 0.3


MATERIALS: dict[str, GearMaterial] = {
    "case_carburized": GearMaterial("20MnCr5 (1.7147) or 18CrNiMo7-6 (1.6587)", "case carburized, hardened and tempered",
                                    "58–62 HRC surface, core 30–42 HRC", 1500.0, 500.0, "case"),
    "nitrided": GearMaterial("31CrMoV9 (1.8519)", "quenched and tempered, gas nitrided",
                             "≥ 650 HV1 surface, NHD per drawing", 1250.0, 420.0, "nitrided"),
    "qt_nitrided": GearMaterial("42CrMo4 (1.7225)", "quenched and tempered, gas nitrided",
                                "≥ 500 HV1 surface, core 28–34 HRC", 1000.0, 360.0, "nitrided"),
    "through_hardened": GearMaterial("42CrMo4 (1.7225)", "quenched and tempered",
                                     "290–330 HB (≈ 315 HV)", 1.313 * 315 + 373, 0.425 * 315 + 187, "through"),
}

## Tooth Form and Stress Correction Factors: Theory

The root stress is greatest at the *critical section*, where a 30° tangent touches the root fillet. The tooth is treated as a cantilever loaded at its tip: the normal load $F_n$ acts along the flank normal at the angle $\alpha_{Fan}$, its bending arm above the section is $h_{Fa}$, the section is $s_{Fn}$ wide and the fillet there has radius $\rho_F$.

In [ ]:
#| echo: false
def _form_figure():
    g = _GG(z=20, m=10.0)
    f = _rat.form_factors(g)
    a = g.alpha
    a_an = np.arccos(g.db / g.da)
    gam = (np.pi / 2 + 2 * g.x * np.tan(a)) / g.z + float(np.tan(a) - a) - float(np.tan(a_an) - a_an)
    aFan = a_an - gam
    ra = g.da / 2
    yL = ra * (np.cos(gam) - np.sin(gam) * np.tan(aFan))   # load line meets the tooth centreline here
    yS = yL - f["h_Fa"]                                     # critical section
    xS = f["s_Fn"] / 2
    fig, ax = plt.subplots(figsize=(11, 6.4))
    P = _rot(gear_outline(g, n_flank=80, n_arc=20), np.pi / 2)
    body = Polygon(P, closed=True, fc=C_PLANET, ec=INK, lw=1.0)
    ax.add_patch(body)
    body.set_clip_path(Wedge((0, 0), ra + 3, 64, 116, width=ra + 3 - (g.df / 2 - 14), transform=ax.transData))
    ax.plot([0, 0], [yS - 14, ra + 8], color=RED, lw=0.5, ls="-.")
    # load at the tip, normal to the flank, at α_Fan to the horizontal
    T = np.array([-ra * np.sin(gam), ra * np.cos(gam)])
    d = np.array([np.cos(aFan), -np.sin(aFan)])
    ax.annotate("", xy=T, xytext=T - 30 * d, arrowprops=dict(arrowstyle="-|>", color=RED, lw=1.8, mutation_scale=14))
    ax.plot([T[0], 0], [T[1], yL], color=RED, lw=0.8, ls="--")
    ax.plot(0, yL, "o", color=RED, ms=4)
    ax.text(T[0] - 31 * d[0] - 1, T[1] - 31 * d[1] + 2, "$F_n$ at the tip, normal to the flank,\nat $\\alpha_{Fan}$ below the horizontal", color=RED, fontsize=9, ha="right")
    ax.plot([T[0] - 30 * d[0], T[0] - 30 * d[0] + 14], [T[1] - 30 * d[1]] * 2, color=INK, lw=0.5)
    # critical section and 30° tangents
    ax.plot([-xS, xS], [yS, yS], color=BLUE, lw=1.6)
    for sg in (-1, 1):
        ax.plot(sg * xS, yS, "o", color=BLUE, ms=4)
        u30 = np.array([sg * np.sin(np.radians(30)), np.cos(np.radians(30))])
        ax.plot([sg * xS - 16 * u30[0], sg * xS + 14 * u30[0]], [yS - 16 * u30[1], yS + 14 * u30[1]], color=GREEN, lw=0.9)
    ax.plot([xS, xS], [yS, yS + 14], color=GREEN, lw=0.5, ls=":")
    ax.add_patch(Arc((xS, yS), 22, 22, theta1=90 - 30, theta2=90, color=GREEN, lw=1.0))
    ax.text(xS + 3.5, yS + 12, "30°", color=GREEN, fontsize=9)
    # dimensions
    _dim(ax, (-xS, yS - 7), (xS, yS - 7), f"$s_{{Fn}}$ = {f['s_Fn']:.1f}", color=BLUE)
    _dim(ax, (32, yS), (32, yL), "", color=BLUE)
    ax.plot([0, 34], [yL, yL], color=INK, lw=0.4, ls=":"); ax.plot([xS, 34], [yS, yS], color=INK, lw=0.4, ls=":")
    ax.text(34, (yS + yL) / 2, f"$h_{{Fa}}$ = {f['h_Fa']:.1f}: bending arm of the\nload above the critical section", color=BLUE, fontsize=9, va="center")
    _label(ax, f"root fillet radius at the section $\\rho_F$ = {f['rho_F']:.1f}", (-xS - 1, yS - 1.5), (-40, yS - 14), color=GREEN, ha="right")
    _label(ax, "critical section: where a 30° tangent\ntouches the fillet on each side", (-xS * 0.5, yS), (-40, yS + 12), color=BLUE, ha="right")
    _label(ax, "30° tangent", (-xS + 10 * 0.5, yS - 10 * 0.866), (-40, yS - 26), color=GREEN, ha="right")
    ax.set_title(f"z = {g.z}, m = {g.m:g} mm (drawn large), x = 0:  $Y_{{Fa}}$ = {f['Y_Fa']:.2f},  $Y_{{Sa}}$ = {f['Y_Sa']:.2f}   "
            "($Y_{Fa} = 6\\,(h_{Fa}/m)\\cos\\alpha_{Fan} / [(s_{Fn}/m)^2\\cos\\alpha]$)", fontsize=10)
    ax.set_xlim(-105, 110); ax.set_ylim(g.df / 2 - 16, ra + 36); ax.set_aspect("equal"); ax.axis("off")
    plt.tight_layout()
_form_figure()

For a gear cut by a rack (hob) with dedendum $h_{fP}$, tip radius $\rho_{fP}$ and no protuberance, ISO 6336-3 gives the section through the auxiliary quantities

$$E = \frac{\pi m}{4} - h_{fP}\tan\alpha - (1-\sin\alpha)\frac{\rho_{fP}}{\cos\alpha},\qquad G = \frac{\rho_{fP}}{m} - \frac{h_{fP}}{m} + x,\qquad H = \frac{2}{z}\left(\frac{\pi}{2} - \frac{E}{m}\right) - \frac{\pi}{3}$$

$$\vartheta = \frac{2G}{z}\tan\vartheta - H\quad\text{(iterate from } \vartheta = \pi/6\text{)}$$

$$\frac{s_{Fn}}{m} = z\sin\left(\frac{\pi}{3}-\vartheta\right) + \sqrt3\left(\frac{G}{\cos\vartheta} - \frac{\rho_{fP}}{m}\right),\qquad \frac{\rho_F}{m} = \frac{\rho_{fP}}{m} + \frac{2G^2}{\cos\vartheta\,(z\cos^2\vartheta - 2G)}.$$

With the load at the tip ($\alpha_{an} = \arccos(d_b/d_a)$, $\gamma_a = \frac{1}{z}(\frac{\pi}{2} + 2x\tan\alpha) + \operatorname{inv}\alpha - \operatorname{inv}\alpha_{an}$, $\alpha_{Fan} = \alpha_{an} - \gamma_a$):

$$\frac{h_{Fa}}{m} = \frac12\left[(\cos\gamma_a - \sin\gamma_a\tan\alpha_{Fan})\frac{d_a}{m} - z\cos\left(\frac{\pi}{3}-\vartheta\right) - \frac{G}{\cos\vartheta} + \frac{\rho_{fP}}{m}\right]$$

$$Y_{Fa} = \frac{6\,(h_{Fa}/m)\cos\alpha_{Fan}}{(s_{Fn}/m)^2\cos\alpha},\qquad Y_{Sa} = (1.2 + 0.13L)\,q_s^{\frac{1}{1.21 + 2.3/L}},\quad L = \frac{s_{Fn}}{h_{Fa}},\ q_s = \frac{s_{Fn}}{2\rho_F}.$$

**Internal gears.** ISO 6336-3 treats the ring tooth as a rack tooth with the ring's own reference thickness $s$, addendum $h_a = m(h_{aP}^* + x - k^*)$, dedendum $h_f = m(h_{fP}^* - x)$ and fillet radius $\rho = \rho_{fP}^* m$. Put the origin on the tooth centreline at the root line, $y$ upwards. The straight flank is $x = C - y\tan\alpha$, and the fillet circle touches both the root line and the flank:

$$C = \frac{s}{2} + h_f\tan\alpha,\qquad y_c = \rho,\qquad x_c = C - \rho\tan\alpha + \frac{\rho}{\cos\alpha}.$$

The 30° tangent touches the fillet at $(x_c - \rho\cos 30°,\ y_c - \rho\sin 30°)$, so

$$s_{Fn} = 2\,(x_c - \rho\cos 30°),\qquad \rho_F = \rho.$$

The tip load acts at the tip corner $(w_a, y_a)$ with $y_a = h_f + h_a$ and $w_a = C - y_a\tan\alpha$, normal to the flank, so it crosses the centreline at $y_a - w_a\tan\alpha$ and

$$h_{Fa} = y_a - w_a\tan\alpha - (y_c - \rho\sin 30°),\qquad \alpha_{Fan} = \alpha.$$

$Y_{Fa}$ and $Y_{Sa}$ then follow from the same two formulas as above. The fillet radius of a shaped ring comes from the shaper cutter tip radius; `GearGeometry.rho_f` of the ring stands for it.

**Rim thickness factor.** A thin rim under the teeth bends as well. With rim thickness $s_R$ and tooth depth $h_t = (h_{aP}^* + h_{fP}^*)\,m$:

In [ ]:
#| echo: false
def _rim_figure():
    fig, (a0, a1) = plt.subplots(1, 2, figsize=(12, 4.4), gridspec_kw=dict(width_ratios=[1, 1.25]))
    # left: what s_R and h_t are
    ge = _GG(z=40, m=10.0)
    s_R = 0.8 * (ge.ha + ge.hf) * ge.m
    r_in = ge.df / 2 - s_R
    P = _rot(gear_outline(ge, n_flank=40, n_arc=12), np.pi / 2)
    body = Polygon(P, closed=True, fc=C_SUN, ec=INK, lw=0.9)
    a0.add_patch(body)
    body.set_clip_path(Wedge((0, 0), ge.da / 2 + 2, 70, 110, width=ge.da / 2 + 2 - r_in, transform=a0.transData))
    a0.add_patch(Arc((0, 0), 2 * r_in, 2 * r_in, theta1=70, theta2=110, color=INK, lw=1.0))
    _dim(a0, (52, r_in), (52, ge.df / 2), "", color=BLUE)
    _dim(a0, (64, ge.df / 2), (64, ge.da / 2), "", color=BLUE)
    for y in (r_in, ge.df / 2, ge.da / 2):
        a0.plot([30, 66], [y, y], color=INK, lw=0.4, ls=":")
    a0.text(55, (r_in + ge.df / 2) / 2, "$s_R$: rim under\nthe tooth roots", color=BLUE, fontsize=9, va="center")
    a0.text(67, (ge.df / 2 + ge.da / 2) / 2, "$h_t$: tooth depth", color=BLUE, fontsize=9, va="center")
    a0.text(0, r_in - 9, "bore, key groove or\nring outside diameter", ha="center", fontsize=8.5, va="top")
    a0.set_xlim(-75, 130); a0.set_ylim(r_in - 26, ge.da / 2 + 6); a0.set_aspect("equal"); a0.axis("off")
    a0.set_title("Rim thickness $s_R$ and tooth depth $h_t$", fontsize=10)
    # right: Y_B curves
    r = np.linspace(0.5, 1.6, 200)
    a1.plot(r, np.where(r < 1.2, 1.6 * np.log(2.242 / r), 1.0), color=BLUE, lw=1.8, label="external gear: $Y_B$ vs $s_R/h_t$")
    q = np.linspace(1.75, 4.5, 200)
    a1.plot(q / 3.5 * 1.2, np.where(q < 3.5, 1.15 * np.log(8.324 / q), 1.0), color=GREEN, lw=1.8, ls="--",
            label="internal gear: $Y_B$ vs $s_R/m$ (top axis)")
    a1.axhline(1, color=INK, lw=0.5)
    a1.set_xlabel("$s_R/h_t$ (external)"); a1.set_ylabel("rim thickness factor $Y_B$")
    sec = a1.secondary_xaxis("top", functions=(lambda x: x * 3.5 / 1.2, lambda y: y * 1.2 / 3.5))
    sec.set_xlabel("$s_R/m$ (internal)")
    a1.text(0.52, 1.02, "below 0.5 $h_t$ (external) or 1.75 m (internal):\noutside ISO 6336-3, rejected", fontsize=8.5, va="bottom")
    a1.legend(fontsize=8.5, frameon=False, loc="upper right")
    a1.spines["right"].set_visible(False)
    plt.tight_layout()
_rim_figure()


$$Y_B = 1.6\ln\frac{2.242\,h_t}{s_R}\ \ (0.5 \le s_R/h_t < 1.2,\ \text{external}),\qquad Y_B = 1.15\ln\frac{8.324\,m}{s_R}\ \ (1.75 \le s_R/m < 3.5,\ \text{internal}),$$

and $Y_B = 1$ for thicker rims. Thinner rims than the lower limits are outside ISO 6336 and are rejected.

In [ ]:
#| export
def _rack_tooth_section(s: float, h_a: float, h_f: float, rho: float, alpha: float) -> tuple[float, float, float]:
    """
    30° tangent section of a straight-sided (rack) tooth: (s_Fn, h_Fa, rho_F) in the units of the inputs.

    s: thickness on the reference line; h_a, h_f: addendum and dedendum from the reference line;
    rho: root fillet radius. Load applied at the tip, normal to the flank.
    """
    C = s / 2 + h_f * math.tan(alpha)                      # flank: x = C − y·tan α  (y up from the root line)
    yc = rho
    xc = C - yc * math.tan(alpha) + rho / math.cos(alpha)  # fillet centre, tangent to root line and flank
    x_pt = xc - rho * math.cos(math.radians(30))
    y_pt = yc - rho * math.sin(math.radians(30))
    y_a = h_f + h_a
    w_a = C - y_a * math.tan(alpha)
    h_Fa = y_a - w_a * math.tan(alpha) - y_pt
    return 2 * x_pt, h_Fa, rho


def form_factors(g: GearGeometry) -> dict[str, float]:
    """
    Tooth form factor Y_Fa, stress correction factor Y_Sa and the critical-section values
    s_Fn, h_Fa, rho_F [mm] for load at the tip (see theory). Works for external and internal gears.
    """
    m, a = g.m, g.alpha
    if g.internal:
        s = g.s
        h_a = g.m * (g.ha + g.x - g.k_tip)
        h_f = g.m * (g.hf - g.x)
        sFn, hFa, rhoF = _rack_tooth_section(s, h_a, h_f, g.rho_f * m, a)
        aFan = a
    else:
        hfP, rfP = g.hf * m, g.rho_f * m
        E = math.pi / 4 * m - hfP * math.tan(a) - (1 - math.sin(a)) * rfP / math.cos(a)
        G = rfP / m - hfP / m + g.x
        H = 2 / g.z * (math.pi / 2 - E / m) - math.pi / 3
        th = math.pi / 6
        for _ in range(100):
            th_new = 2 * G / g.z * math.tan(th) - H
            if abs(th_new - th) < 1e-12:
                break
            th = th_new
        sFn = m * (g.z * math.sin(math.pi / 3 - th) + math.sqrt(3) * (G / math.cos(th) - rfP / m))
        rhoF = m * (rfP / m + 2 * G ** 2 / (math.cos(th) * (g.z * math.cos(th) ** 2 - 2 * G)))
        a_an = math.acos(g.db / g.da)
        gam = (math.pi / 2 + 2 * g.x * math.tan(a)) / g.z + float(inv(a)) - float(inv(a_an))
        aFan = a_an - gam
        hFa = m / 2 * ((math.cos(gam) - math.sin(gam) * math.tan(aFan)) * g.da / m
                       - g.z * math.cos(math.pi / 3 - th) - G / math.cos(th) + rfP / m)
    Y_Fa = 6 * (hFa / m) * math.cos(aFan) / ((sFn / m) ** 2 * math.cos(a))
    L = sFn / hFa
    qs = sFn / (2 * rhoF)
    Y_Sa = (1.2 + 0.13 * L) * qs ** (1 / (1.21 + 2.3 / L))
    return {"Y_Fa": Y_Fa, "Y_Sa": Y_Sa, "s_Fn": sFn, "h_Fa": hFa, "rho_F": rhoF}


def rim_factor(g: GearGeometry, s_R: float | None) -> float:
    "ISO 6336-3 rim thickness factor Y_B for rim thickness s_R [mm] (None: solid gear, Y_B = 1)."
    if s_R is None:
        return 1.0
    if g.internal:
        r = s_R / g.m
        if r >= 3.5:
            return 1.0
        if r < 1.75:
            raise ValueError(f"{g.name}: rim s_R = {s_R:.2f} mm < 1.75 m is outside ISO 6336-3")
        return 1.15 * math.log(8.324 * g.m / s_R)
    ht = (g.ha + g.hf) * g.m
    r = s_R / ht
    if r >= 1.2:
        return 1.0
    if r < 0.5:
        raise ValueError(f"{g.name}: rim s_R = {s_R:.2f} mm < 0.5 h_t is outside ISO 6336-3")
    return 1.6 * math.log(2.242 * ht / s_R)

In [ ]:
for z in (12, 17, 20, 30, 60, 200):
    f = form_factors(GearGeometry(z=z, m=1, x=0.0 if z >= 17 else 0.4))
    print(f"z = {z:3d}: Y_Fa = {f['Y_Fa']:.3f}  Y_Sa = {f['Y_Sa']:.3f}  Y_Fa·Y_Sa = {f['Y_Fa'] * f['Y_Sa']:.3f}")
f20 = form_factors(GearGeometry(z=20, m=1))
assert 2.7 < f20["Y_Fa"] < 2.9 and 1.5 < f20["Y_Sa"] < 1.6         # ISO 6336-3 chart values ≈ 2.80 / 1.55
# the rack-tooth model used for internal gears agrees with the external formulas as z → ∞
fe = form_factors(GearGeometry(z=50000, m=1))
fr = _rack_tooth_section(math.pi / 2, 1.0, 1.25, 0.38, math.radians(20))
assert abs(fe["s_Fn"] - fr[0]) < 1e-3 and abs(fe["h_Fa"] - fr[1]) < 1e-3
print("internal ring z = 72:", {k: round(v, 3) for k, v in form_factors(GearGeometry(z=72, m=1, internal=True, k_tip=0.2)).items()})

## Load Factors: Theory

**Load sharing between planets, $K_\gamma$.** Tolerances on pin positions and tooth thickness make one planet carry more than $1/N$ of the torque. Typical values for a sun held on a stiff shaft (after AGMA 6123-B06):

| $N$ | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
|---|---|---|---|---|---|---|---|
| $K_\gamma$ | 1.00 | 1.10 | 1.25 | 1.35 | 1.44 | 1.47 | 1.50 |

Three planets with a floating sun (radially free, for example on a crowned spline or a flexible coupling) share the load evenly: $K_\gamma = 1.0$.

**Dynamic factor $K_v$** (ISO 6336-1 Method C for spur gears, using the pitch line velocity relative to the carrier, $v = \pi d_s |n_s - n_c|/60\,000$):

$$K_v = 1 + \left(\frac{K_1}{K_A F_t/b} + K_2\right)\frac{z_1 v}{100}\sqrt{\frac{u^2}{1+u^2}},\qquad K_2 = 0.0193,$$

with $K_A F_t/b \ge 100$ N/mm (smaller values are raised to 100), $z_1$ the smaller gear and $u = z_{large}/z_{small}$ for the sun–planet mesh, and $z_1 = z_p$, $u = z_r/z_p$ for the planet–ring mesh. $K_1$ comes from the ISO 1328 accuracy grade:

| grade | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 |
|---|---|---|---|---|---|---|---|---|
| $K_1$ | 2.1 | 3.9 | 7.5 | 14.9 | 26.8 | 39.1 | 52.8 | 76.6 |

**Face load factor.** The planet pins bend and the carrier twists, so the load is not uniform across the face. $K_{H\beta}$ is an input (default 1.2, a reasonable value for a straddle-mounted double-plate carrier with crowned or ground planets; use 1.35 or more for a single-plate cantilever carrier). The bending factor follows from it:

$$K_{F\beta} = K_{H\beta}^{N_F},\qquad N_F = \frac{(b/h)^2}{1 + b/h + (b/h)^2},\qquad h = (h_{aP}^* + h_{fP}^*)\,m,$$

with $b$ the smaller face width of the mesh.

**Transverse load factors** (spur, $K_A F_t/b \ge 100$ N/mm): $K_{H\alpha} = K_{F\alpha} = 1.0$ for grades 5–6; for surface-hardened gears 1.1 at grade 7 and 1.2 at grade 8; for through-hardened gears 1.0 at grade 7 and 1.1 at grade 8. The surface-hardened line is used unless one of the three gears is through hardened.

**Application factor $K_A$** is yours to choose: 1.0 for a smooth electric motor driving a uniform load, 1.25 for moderate shocks, 1.5–1.75 for heavy shocks. If you rate on peak torque instead of the nominal torque, use $K_A = 1$ with the peak torque, and check the static strength separately.

In [ ]:
#| export
K_GAMMA = {1: 1.0, 2: 1.0, 3: 1.10, 4: 1.25, 5: 1.35, 6: 1.44, 7: 1.47, 8: 1.50}
_K1_SPUR = {3: 2.1, 4: 3.9, 5: 7.5, 6: 14.9, 7: 26.8, 8: 39.1, 9: 52.8, 10: 76.6, 11: 102.6, 12: 146.3}


def load_sharing_factor(n_planets: int, floating_sun: bool = False) -> float:
    "Mesh load factor K_γ for N planets (see table)."
    if floating_sun and n_planets == 3:
        return 1.0
    return K_GAMMA.get(n_planets, 1.5)


def dynamic_factor(z1: int, u: float, v: float, wt: float, grade: int) -> float:
    "K_v (ISO 6336-1 Method C, spur). z1 pinion teeth, u = |z2/z1|, v [m/s], wt = K_A·F_t/b [N/mm]."
    K1 = _K1_SPUR[int(min(max(grade, 3), 12))]
    wt = max(wt, 100.0)
    return 1 + (K1 / wt + 0.0193) * z1 * v / 100 * math.sqrt(u * u / (1 + u * u))


def transverse_factor(grade: int, surface_hardened: bool = True) -> float:
    "K_Hα = K_Fα for spur gears with K_A·F_t/b ≥ 100 N/mm."
    if grade <= 6:
        return 1.0
    if surface_hardened:
        return {7: 1.1, 8: 1.2}.get(grade, 1.4)
    return {7: 1.0, 8: 1.1}.get(grade, 1.2)

## Life Factors: Theory

A gear that sees fewer load cycles than its endurance limit can take more stress. ISO 6336-2/-3 Method B give log–log straight lines through these points; above $10^{10}$ cycles the factor is held. This module uses the conservative branch (factor falls to 0.85 at $10^{10}$ cycles), which applies unless the lubrication, material cleanliness and manufacturing are all known to be optimal.

| kind | $Z_{NT}$: static / knee | $Y_{NT}$: static / knee |
|---|---|---|
| `case`, `through` | 1.6 at ≤ $10^5$, 1.0 at $5\cdot10^7$ | 2.5 at ≤ $10^3$ (through: $10^4$), 1.0 at $3\cdot10^6$ |
| `nitrided` | 1.3 at ≤ $10^5$, 1.0 at $2\cdot10^6$ | 1.6 at ≤ $10^3$, 1.0 at $3\cdot10^6$ |

Between the points the factor is interpolated on log–log scales: for $N_0 \le N \le N_1$ with factors $f_0, f_1$,

$$\log f = \log f_0 + \frac{\log N - \log N_0}{\log N_1 - \log N_0}\,(\log f_1 - \log f_0).$$

In [ ]:
#| echo: false
def _life_figure():
    N = np.logspace(3, 11, 300)
    fig, (a0, a1) = plt.subplots(1, 2, figsize=(12, 4))
    for kind, col, ls in (("case", BLUE, "-"), ("nitrided", GREEN, "--"), ("through", RED, ":")):
        a0.plot(N, [_rat.life_factor_contact(n, kind) for n in N], color=col, ls=ls, lw=1.8, label=kind)
        a1.plot(N, [_rat.life_factor_bending(n, kind) for n in N], color=col, ls=ls, lw=1.8, label=kind)
    for ax, nm in ((a0, "contact life factor $Z_{NT}$"), (a1, "bending life factor $Y_{NT}$")):
        ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("load cycles $N_L$"); ax.set_ylabel(nm)
        ax.set_yticks([0.85, 1, 1.3, 1.6, 2.5]); ax.set_yticklabels(["0.85", "1.0", "1.3", "1.6", "2.5"])
        ax.yaxis.set_minor_formatter(plt.NullFormatter())
        ax.axhline(1, color=INK, lw=0.4)
        ax.legend(fontsize=8.5, frameon=False)
        for sp in ("top", "right"):
            ax.spines[sp].set_visible(False)
    a0.set_title("Contact (pitting)", fontsize=10); a1.set_title("Bending (root breakage)", fontsize=10)
    plt.tight_layout()
_life_figure()


**Load cycles in a planetary set** (ring fixed, $L_h$ hours, one flank loaded per mesh):

$$N_{L,sun} = 60\,L_h\,|n_s - n_c|\,N,\qquad N_{L,planet} = 60\,L_h\,|n_{p/c}|,\qquad N_{L,ring} = 60\,L_h\,|n_c|\,N.$$

The planet meshes with the sun on one flank and with the ring on the other, so each tooth root is bent one way and then the other on every revolution. ISO 6336-3 accounts for this alternating bending with the mean stress factor $Y_M = 0.7$ on the planet.

In [ ]:
#| export
def _loglog(N: float, pts: list[tuple[float, float]]) -> float:
    if N <= pts[0][0]:
        return pts[0][1]
    for (n0, f0), (n1, f1) in zip(pts, pts[1:]):
        if N <= n1:
            t = (math.log10(N) - math.log10(n0)) / (math.log10(n1) - math.log10(n0))
            return 10 ** (math.log10(f0) + t * (math.log10(f1) - math.log10(f0)))
    return pts[-1][1]


def life_factor_contact(N: float, kind: str) -> float:
    "Z_NT (ISO 6336-2 Method B, conservative branch above the knee)."
    if kind == "nitrided":
        return _loglog(N, [(1e5, 1.3), (2e6, 1.0), (1e10, 0.85)])
    return _loglog(N, [(1e5, 1.6), (5e7, 1.0), (1e10, 0.85)])


def life_factor_bending(N: float, kind: str) -> float:
    "Y_NT (ISO 6336-3 Method B, conservative branch above the knee)."
    if kind == "nitrided":
        return _loglog(N, [(1e3, 1.6), (3e6, 1.0), (1e10, 0.85)])
    if kind == "through":
        return _loglog(N, [(1e4, 2.5), (3e6, 1.0), (1e10, 0.85)])
    return _loglog(N, [(1e3, 2.5), (3e6, 1.0), (1e10, 0.85)])

In [ ]:
assert life_factor_contact(1e4, "case") == 1.6 and math.isclose(life_factor_contact(5e7, "case"), 1.0)
assert math.isclose(life_factor_bending(3e6, "case"), 1.0) and life_factor_bending(1e11, "case") == 0.85

## Contact and Bending Stress: Theory

The force along the line of action is the same at both meshes of a planet (moment balance about the pin), and its component on the reference circle is the nominal tangential force. The carrier holds the pin against the sum of the two.

In [ ]:
#| echo: false
def _forces_figure():
    ps = _lay.PlanetarySet.build(18, 27, 72, m=2.0, n_planets=3, b=20)
    fig, ax = plt.subplots(figsize=(11, 7))
    c = np.array([0.0, ps.a_w])
    rs = ps.a_w * ps.sun.z / (ps.sun.z + ps.planet.z)
    # sun, one planet and a piece of the ring (no teeth: forces only)
    ax.add_patch(Circle((0, 0), rs, fc=C_SUN, ec=INK, lw=0.9))
    ax.add_patch(Circle(c, ps.a_w - rs, fc=C_PLANET, ec=INK, lw=0.9))
    ax.add_patch(Arc((0, 0), 2 * (2 * ps.a_w - rs), 2 * (2 * ps.a_w - rs), theta1=55, theta2=125, color=INK, lw=1.2))
    ax.add_patch(Circle(c, 4, fc=C_STEEL, ec=INK, lw=0.8))
    A, B = np.array([0, rs]), np.array([0, 2 * ps.a_w - rs])
    aw = ps.sun_planet.alpha_w
    L = 26
    # forces ON THE PLANET: from the sun at A and from the ring at B, both along their lines of action
    dA = np.array([-np.cos(aw), np.sin(aw)])         # sun (counter-clockwise) drives the planet to the left, radial part away from the sun
    dB = np.array([-np.cos(aw), -np.sin(aw)])        # ring reaction also to the left, radial part towards the planet centre
    for P_, d_, nm, off in ((A, dA, "$F_{bn}$ from the sun", (-2, 2)), (B, dB, "$F_{bn}$ from the ring", (-2, -7))):
        ax.annotate("", xy=P_ + L * d_, xytext=P_, arrowprops=dict(arrowstyle="-|>", color=RED, lw=1.8, mutation_scale=14))
        ax.plot([P_[0] - 16 * d_[0], P_[0] + (L + 10) * d_[0]], [P_[1] - 16 * d_[1], P_[1] + (L + 10) * d_[1]], color=GREEN, lw=0.6, ls="--")
        ax.text(*(P_ + L * d_ + off), nm, color=RED, fontsize=9, ha="right")
        ax.annotate("", xy=P_ + [-L * np.cos(aw), 0], xytext=P_, arrowprops=dict(arrowstyle="-|>", color=BLUE, lw=1.2))
    ax.text(A[0] - 14, A[1] - 6, "$F_t$", color=BLUE, fontsize=10)
    ax.text(B[0] - 14, B[1] + 2, "$F_t$", color=BLUE, fontsize=10)
    # pin reaction: 2 F_t to the left
    ax.annotate("", xy=c + [2 * L * np.cos(aw), 0], xytext=c, arrowprops=dict(arrowstyle="-|>", color=INK, lw=1.8, mutation_scale=14))
    ax.text(c[0] + 2 * L * np.cos(aw) + 2, c[1] + 2, "carrier force on the pin $\\approx 2F_t$\n(radial parts cancel)", fontsize=9)
    for P_, nm in ((A, "A: sun–planet pitch point"), (B, "B: planet–ring pitch point")):
        ax.plot(*P_, "o", color=INK, ms=3.5)
    _label(ax, "A: sun–planet pitch point", A, (-75, 12), ha="right")
    _label(ax, "B: planet–ring pitch point", B, (-75, B[1] + 12), ha="right")
    _label(ax, "lines of action at $\\alpha_w$", A - 12 * dA, (-75, 28), color=GREEN, ha="right")
    _label(ax, "sun: $T_s = T_{out}/(i\\,\\eta)$", (-rs * 0.5, rs * 0.4), (-75, -2), ha="right")
    _label(ax, "planet pin, held by the carrier:\n$T_{out} = N\\cdot 2F_t\\cdot a_w$ (approx.)", c + [-3, -3], (-75, 46), ha="right")
    _label(ax, "ring (fixed): $T_r = -(T_{out} - T_s)$", B[1] * np.array([np.cos(np.radians(124)), np.sin(np.radians(124))]), (-75, 64), ha="right")
    ax.text(62, rs + 4, "$F_t = \\dfrac{2000\\,T_s}{N\\,d_s}$\n\n$F_{bn} = \\dfrac{F_t}{\\cos\\alpha}$\n\nsame on both meshes\n(moment balance about the pin)",
            fontsize=10, va="center")
    ax.set_xlim(-160, 140); ax.set_ylim(-8, B[1] + 32); ax.set_aspect("equal"); ax.axis("off")
    ax.set_title("Forces on one planet (sun turning counter-clockwise, ring fixed)", fontsize=10)
    plt.tight_layout()
_forces_figure()

$$T_s = \frac{T_{out}}{i\,\eta},\qquad F_t = \frac{2000\,T_s}{N\,d_s}\quad[\text{N, with } T_s \text{ in N·m and } d_s \text{ in mm}],\qquad F_{bn} = \frac{F_t}{\cos\alpha},\qquad T_r = -(T_{out} - T_s).$$

$\eta$ is the stage efficiency (default 0.97) and $T_r$ the reaction torque the housing must hold. The pitch line velocity used for $K_v$ is taken relative to the carrier, $v = \pi d_s\,\lvert n_s - n_c\rvert/60\,000$ m/s.

**Contact (ISO 6336-2)**, $u = z_2/z_1 > 0$, upper sign external, lower sign internal:

$$\sigma_{H0} = Z_H Z_E Z_\varepsilon\sqrt{\frac{F_t}{d_1 b}\,\frac{u \pm 1}{u}},\qquad \sigma_{H1,2} = Z_{B,D}\,\sigma_{H0}\sqrt{K_A K_v K_{H\beta} K_{H\alpha} K_\gamma}$$

$$Z_H = \sqrt{\frac{2\cos\alpha_w}{\cos^2\alpha\,\sin\alpha_w}},\qquad Z_E = \sqrt{\frac{1}{\pi\left(\frac{1-\nu_1^2}{E_1} + \frac{1-\nu_2^2}{E_2}\right)}},\qquad Z_\varepsilon = \sqrt{\frac{4-\varepsilon_\alpha}{3}}.$$

$Z_B$, $Z_D$ move the stress from the pitch point to the inner point of single tooth contact of an external pinion and wheel, where one tooth pair carries the whole load:

In [ ]:
#| echo: false
def _contact_figure():
    ps = _lay.PlanetarySet.build(18, 27, 72, m=2.0, n_planets=3, b=20)
    g1, g2, mc = ps.sun, ps.planet, ps.sun_planet
    pb = g1.pb
    ga = mc.eps_alpha * pb                       # path of contact AE
    rb1, rb2 = g1.db / 2, g2.db / 2
    t2 = np.sqrt((g2.da / 2) ** 2 - rb2 ** 2)
    AC = mc.a_w * np.sin(mc.alpha_w) - t2        # T1A; C at T1C = r_b1 tan α_w
    T1C = rb1 * np.tan(mc.alpha_w)
    xA, xE = 0.0, ga
    xC = T1C - AC
    xB, xD = xE - pb, xA + pb
    fig, ax = plt.subplots(figsize=(11, 3.6))
    ax.fill_between([xA, xB], 0, 0.5, color=C_PLANET); ax.fill_between([xB, xD], 0, 1.0, color="#f3c9a3")
    ax.fill_between([xD, xE], 0, 0.5, color=C_PLANET)
    ax.plot([xA, xA, xB, xB, xD, xD, xE, xE], [0, 0.5, 0.5, 1, 1, 0.5, 0.5, 0], color=INK, lw=1.2)
    for x, nm in ((xA, "A"), (xB, "B"), (xC, "C"), (xD, "D"), (xE, "E")):
        ax.plot([x, x], [-0.08, 0], color=INK, lw=1); ax.text(x, -0.18, nm, ha="center", fontsize=11, color=RED if nm == "C" else INK)
    ax.text((xA + xB) / 2, 0.25, "two pairs\nin contact", ha="center", va="center", fontsize=8.5)
    ax.text((xB + xD) / 2, 0.55, "one pair carries\nthe whole load", ha="center", va="center", fontsize=8.5)
    ax.text((xD + xE) / 2, 0.25, "two pairs\nin contact", ha="center", va="center", fontsize=8.5)
    _dim(ax, (xA, 1.25), (xE, 1.25), f"path of contact $g_\\alpha$ = AE = $\\varepsilon_\\alpha p_b$ = {ga:.2f} mm ($\\varepsilon_\\alpha$ = {mc.eps_alpha:.2f})", color=INK)
    _dim(ax, (xA, -0.4), (xD, -0.4), "$p_b$", color=BLUE); _dim(ax, (xB, -0.62), (xE, -0.62), "$p_b$", color=BLUE)
    _label(ax, "B: inner point of single contact on the sun\n($Z_B$ moves $\\sigma_H$ from C to here)", (xB, 1.0), (xE + 1.2, 0.9))
    _label(ax, "D: inner point of single contact on the planet ($Z_D$)", (xD, 0.75), (xE + 1.2, 0.45))
    _label(ax, "C: pitch point, where $\\sigma_{H0}$ is defined", (xC, -0.1), (xE + 1.2, 0.05), color=RED)
    ax.set_ylabel("share of the\ntooth load")
    ax.set_xlim(-0.5, xE + 13); ax.set_ylim(-0.8, 1.45); ax.set_yticks([0, 0.5, 1]); ax.set_xticks([])
    for sp in ("top", "right", "bottom"):
        ax.spines[sp].set_visible(False)
    ax.set_title("Load along the path of contact of the sun–planet mesh (18/27, m = 2)", fontsize=10)
    plt.tight_layout()
_contact_figure()

$Z_{B} = \max(1, M_1)$, $Z_D = \max(1, M_2)$:

$$M_1 = \frac{\tan\alpha_w}{\sqrt{\left(\sqrt{\frac{d_{a1}^2}{d_{b1}^2}-1} - \frac{2\pi}{z_1}\right)\left(\sqrt{\frac{d_{a2}^2}{d_{b2}^2}-1} - (\varepsilon_\alpha - 1)\frac{2\pi}{z_2}\right)}},$$

$M_2$ with indices 1 and 2 swapped. For the internal mesh both are taken as 1.

Permissible contact stress and safety factor:

$$\sigma_{HP} = \frac{\sigma_{H\,lim} Z_{NT} Z_{LvR} Z_W Z_X}{S_{H\,min}},\qquad S_H = \frac{\sigma_{H\,lim} Z_{NT} Z_{LvR} Z_W Z_X}{\sigma_H},$$

with $Z_{LvR}$ = 0.85 for hobbed or shaped flanks ($R_z > 4\,\mu$m), 0.92 for ground or shaved flanks, $Z_W = Z_X = 1$.

**Bending (ISO 6336-3)**, for each gear of the mesh with its own form factors:

$$\sigma_F = \frac{F_t}{b\,m}\,Y_{Fa} Y_{Sa} Y_\varepsilon Y_B\;K_A K_v K_{F\beta} K_{F\alpha} K_\gamma,\qquad Y_\varepsilon = 0.25 + \frac{0.75}{\varepsilon_\alpha}$$

$$S_F = \frac{\sigma_{F\,lim}\,Y_{ST}\,Y_{NT}\,Y_{\delta relT}\,Y_{RrelT}\,Y_X\,Y_M}{\sigma_F},\qquad Y_{ST} = 2,\ Y_{\delta relT} = Y_{RrelT} = Y_X = 1.$$

with $Y_M = 0.7$ for the planets (alternating bending) and 1 for sun and ring. The permissible bending stress reported is

$$\sigma_{FP} = \frac{\sigma_{F\,lim}\,Y_{ST}\,Y_{NT}\,Y_M}{S_{F\,min}}.$$

Usual minimum safety factors: $S_{H\,min} = 1.0$ to 1.3, $S_{F\,min} = 1.4$ to 2.0 (root breakage is a sudden failure, pitting a gradual one).

**Result.** Each gear is rated in each of its meshes (sun, planet at the sun, planet at the ring, ring). The set passes when

$$\min S_H \ge S_{H\,min}\quad\text{and}\quad \min S_F \ge S_{F\,min},$$

and the *limiting* gear and failure mode is the one with the smallest margin $S_H/S_{H\,min}$ or $S_F/S_{F\,min}$.

In [ ]:
#| export
@dataclass(frozen=True)
class RatingInputs:
    """
    Operating conditions and rating choices.

    torque_out   : float  Output (carrier) torque [N·m] used for the rating
    n_in         : float  Input (sun) speed [rpm]
    life_h       : float  Required life [h] at this torque and speed
    K_A          : float  Application factor
    K_Hbeta      : float  Face load factor (see theory)
    grade        : int    ISO 1328-1 flank tolerance class of the gears
    floating_sun : bool   Sun radially free (three planets share load evenly)
    efficiency   : float  Mesh efficiency of the stage, for the sun torque
    S_Hmin       : float  Required safety against pitting
    S_Fmin       : float  Required safety against tooth breakage
    flank_finish : str    'ground' or 'hobbed' for sun and planets; the ring is always shaped (Z_LvR = 0.85)
    """
    torque_out: float
    n_in: float
    life_h: float = 10_000.0
    K_A: float = 1.25
    K_Hbeta: float = 1.2
    grade: int = 6
    floating_sun: bool = False
    efficiency: float = 0.97
    S_Hmin: float = 1.0
    S_Fmin: float = 1.4
    flank_finish: str = "ground"


@dataclass(frozen=True)
class GearRating:
    "Stresses and safety factors of one gear in one mesh. Stresses in MPa."
    gear: str
    mesh: str
    cycles: float
    sigma_H: float
    sigma_HP: float
    S_H: float
    sigma_F: float
    sigma_FP: float
    S_F: float
    Y_Fa: float
    Y_Sa: float
    Y_B: float
    Z_NT: float
    Y_NT: float


@dataclass(frozen=True)
class PlanetaryRating:
    """
    Result of `rate_planetary`.

    T_sun, T_ring, T_carrier : torques [N·m] (ring reaction is opposite in sign to the others)
    F_t                      : nominal tangential force per planet mesh [N] (before K_γ)
    F_bn                     : force along the line of action per mesh [N] (before K_γ)
    v                        : pitch line velocity relative to the carrier [m/s]
    factors                  : the load factors used
    gears                    : one GearRating per gear and mesh (sun, planet@sun, planet@ring, ring)
    """
    T_sun: float
    T_ring: float
    T_carrier: float
    F_t: float
    F_bn: float
    v: float
    factors: dict
    gears: tuple[GearRating, ...]
    S_Hmin: float
    S_Fmin: float

    @property
    def S_H(self) -> float: return min(r.S_H for r in self.gears)

    @property
    def S_F(self) -> float: return min(r.S_F for r in self.gears)

    @property
    def ok(self) -> bool: return self.S_H >= self.S_Hmin and self.S_F >= self.S_Fmin

    @property
    def limiting(self) -> str:
        "Which gear and failure mode has the lowest margin over its required safety."
        rows = [(r.S_H / self.S_Hmin, f"pitting of the {r.gear} ({r.mesh} mesh)") for r in self.gears]
        rows += [(r.S_F / self.S_Fmin, f"root breakage of the {r.gear} ({r.mesh} mesh)") for r in self.gears]
        return min(rows)[1]

    def table(self) -> str:
        "Fixed-width text table of the results."
        hdr = f"{'gear':8s} {'mesh':12s} {'cycles':>9s} {'σH':>7s} {'σHP':>7s} {'S_H':>6s} {'σF':>7s} {'σFP':>7s} {'S_F':>6s} {'Y_Fa':>5s} {'Y_Sa':>5s} {'Y_B':>5s}"
        lines = [hdr]
        for r in self.gears:
            lines.append(f"{r.gear:8s} {r.mesh:12s} {r.cycles:9.2e} {r.sigma_H:7.0f} {r.sigma_HP:7.0f} {r.S_H:6.2f} "
                         f"{r.sigma_F:7.0f} {r.sigma_FP:7.0f} {r.S_F:6.2f} {r.Y_Fa:5.2f} {r.Y_Sa:5.2f} {r.Y_B:5.2f}")
        lines.append(f"min S_H = {self.S_H:.2f} (need {self.S_Hmin}), min S_F = {self.S_F:.2f} (need {self.S_Fmin}); "
                     f"limited by {self.limiting}")
        return "\n".join(lines)


def _Z_E(m1: GearMaterial, m2: GearMaterial) -> float:
    return math.sqrt(1 / (math.pi * ((1 - m1.nu ** 2) / m1.E + (1 - m2.nu ** 2) / m2.E)))


def _single_pair_factors(g1: GearGeometry, g2: GearGeometry, aw: float, eps: float) -> tuple[float, float]:
    "Z_B, Z_D of an external spur pair (ISO 6336-2)."
    t1 = math.sqrt((g1.da / g1.db) ** 2 - 1)
    t2 = math.sqrt((g2.da / g2.db) ** 2 - 1)
    M1 = math.tan(aw) / math.sqrt((t1 - 2 * math.pi / g1.z) * (t2 - (eps - 1) * 2 * math.pi / g2.z))
    M2 = math.tan(aw) / math.sqrt((t2 - 2 * math.pi / g2.z) * (t1 - (eps - 1) * 2 * math.pi / g1.z))
    return max(1.0, M1), max(1.0, M2)


def rate_planetary(ps: PlanetarySet, inp: RatingInputs,
                   materials: dict[str, GearMaterial] | None = None,
                   rims: dict[str, float | None] | None = None) -> PlanetaryRating:
    """
    ISO 6336 rating of both meshes of a planetary set.

    materials : {'sun', 'planet', 'ring'} → GearMaterial (default: carburized sun and planets, Q&T nitrided ring)
    rims      : {'sun', 'planet', 'ring'} → rim thickness s_R under the tooth roots [mm] (None or missing: solid)
    """
    mats = {"sun": MATERIALS["case_carburized"], "planet": MATERIALS["case_carburized"],
            "ring": MATERIALS["qt_nitrided"]}
    mats.update(materials or {})
    rims = rims or {}
    N = ps.n_planets
    sun, pl, ring = ps.sun, ps.planet, ps.ring
    T_s = inp.torque_out / (ps.ratio * inp.efficiency)
    F_t = 2000 * T_s / (N * sun.d)
    F_bn = F_t / math.cos(sun.alpha)
    sp = ps.speeds(inp.n_in)
    n_rel = sp["sun"] - sp["carrier"]
    v = math.pi * sun.d * abs(n_rel) / 60_000
    Kg = load_sharing_factor(N, inp.floating_sun)
    surf = all(mats[k].kind != "through" for k in mats)
    KHa = transverse_factor(inp.grade, surf)
    cycles = {"sun": 60 * inp.life_h * abs(n_rel) * N, "planet": 60 * inp.life_h * abs(sp["planet_rel"]),
              "ring": 60 * inp.life_h * abs(sp["carrier"]) * N}
    zlvr = {"ground": 0.92, "hobbed": 0.85}[inp.flank_finish]
    ZLvR = {"sun": zlvr, "planet": zlvr, "ring": 0.85}
    factors = {"K_A": inp.K_A, "K_gamma": Kg, "K_Halpha": KHa, "K_Hbeta": inp.K_Hbeta}
    out = []
    for mesh, g1, g2, mc in (("sun–planet", sun, pl, ps.sun_planet), ("planet–ring", pl, ring, ps.planet_ring)):
        u = g2.z / g1.z
        z1 = min(g1.z, g2.z) if not g2.internal else g1.z
        uu = max(u, 1 / u) if not g2.internal else u
        Kv = dynamic_factor(z1, uu, v, inp.K_A * F_t / g1.b, inp.grade)
        b = min(g1.b, g2.b)
        h = (g1.ha + g1.hf) * g1.m
        NF = (b / h) ** 2 / (1 + b / h + (b / h) ** 2)
        KFb = inp.K_Hbeta ** NF
        aw, eps = mc.alpha_w, mc.eps_alpha
        ZH = math.sqrt(2 * math.cos(aw) / (math.cos(g1.alpha) ** 2 * math.sin(aw)))
        k1, k2 = mesh.split("–")
        ZE = _Z_E(mats[k1], mats[k2])
        Ze = math.sqrt((4 - eps) / 3)
        sign = -1 if g2.internal else 1
        sH0 = ZH * ZE * Ze * math.sqrt(F_t / (g1.d * b) * (u + sign) / u)
        ZB, ZD = (1.0, 1.0) if g2.internal else _single_pair_factors(g1, g2, aw, eps)
        KH = inp.K_A * Kv * inp.K_Hbeta * KHa * Kg
        KF = inp.K_A * Kv * KFb * KHa * Kg
        Ye = 0.25 + 0.75 / eps
        factors[f"K_v {mesh}"] = Kv
        factors[f"K_Fbeta {mesh}"] = KFb
        for key, g, Zbd in ((k1, g1, ZB), (k2, g2, ZD)):
            mat = mats[key]
            sH = Zbd * sH0 * math.sqrt(KH)
            ZNT = life_factor_contact(cycles[key], mat.kind)
            sHlim_eff = mat.sigma_Hlim * ZNT * ZLvR[key]
            ff = form_factors(g)
            YB = rim_factor(g, rims.get(key))
            sF = F_t / (b * g.m) * ff["Y_Fa"] * ff["Y_Sa"] * Ye * YB * KF
            YNT = life_factor_bending(cycles[key], mat.kind)
            YM = 0.7 if key == "planet" else 1.0
            sFlim_eff = mat.sigma_Flim * 2.0 * YNT * YM
            out.append(GearRating(key, mesh, cycles[key], sH, sHlim_eff / inp.S_Hmin, sHlim_eff / sH,
                                  sF, sFlim_eff / inp.S_Fmin, sFlim_eff / sF,
                                  ff["Y_Fa"], ff["Y_Sa"], YB, ZNT, YNT))
    T_c = inp.torque_out
    return PlanetaryRating(T_s, -(T_c - T_s), T_c, F_t, F_bn, v, factors, tuple(out), inp.S_Hmin, inp.S_Fmin)

### Example

The ratio-5 set of the layout notebook (module 1.5, 18/27/72, three planets, 20 mm face) at 40 N·m output, 3000 rpm input, 10 000 h:

In [ ]:
from emachines.gears.layout import PlanetarySet
ps = PlanetarySet.build(18, 27, 72, m=1.5, n_planets=3, b=20)
r = rate_planetary(ps, RatingInputs(torque_out=40.0, n_in=3000.0, life_h=10_000))
print(f"T_sun = {r.T_sun:.2f} N·m, T_ring = {r.T_ring:.2f} N·m, F_t = {r.F_t:.0f} N per planet, v = {r.v:.2f} m/s")
print({k: round(v, 3) for k, v in r.factors.items()})
print(r.table())
assert r.ok

Doubling the torque roughly halves the bending safety and divides the pitting safety by $\sqrt2$, which is the scaling to expect:

In [ ]:
r2 = rate_planetary(ps, RatingInputs(torque_out=80.0, n_in=3000.0, life_h=10_000))
print(f"S_H {r.S_H:.2f} → {r2.S_H:.2f}, S_F {r.S_F:.2f} → {r2.S_F:.2f}")
assert 0.45 < r2.S_F / r.S_F < 0.55 and 0.65 < r2.S_H / r.S_H < 0.75

---

## Tests

In [ ]:
#| hide
# Z_E for steel/steel is 189.8 √MPa
assert abs(_Z_E(MATERIALS["case_carburized"], MATERIALS["case_carburized"]) - 189.8) < 0.1
# Z_H at α_w = 20° is 2.495
aw = math.radians(20)
assert abs(math.sqrt(2 * math.cos(aw) / (math.cos(aw) ** 2 * math.sin(aw))) - 2.495) < 1e-3
# K_v grows with speed and accuracy class
assert dynamic_factor(18, 1.5, 10, 200, 8) > dynamic_factor(18, 1.5, 10, 200, 5) > dynamic_factor(18, 1.5, 1, 200, 5) > 1
# rim factor continuity at the limits
g_ext, g_int = GearGeometry(z=20, m=2), GearGeometry(z=80, m=2, internal=True)
assert abs(rim_factor(g_ext, 1.2 * 4.5) - 1.0) < 2e-3 and abs(rim_factor(g_int, 3.5 * 2) - 1.0) < 2e-3
# the sun, sharing its mesh with three planets, sees N times more cycles per relative turn than the ring
cyc = {g.gear: g.cycles for g in r.gears}
assert math.isclose(cyc["sun"] / cyc["ring"], (3000 - 600) / 600)
# ring torque balances the others
assert math.isclose(r.T_sun + r.T_ring, r.T_carrier * 0 + r.T_sun - (r.T_carrier - r.T_sun))
# a floating sun with three planets lowers K_γ to 1 and raises the safety
r3 = rate_planetary(ps, RatingInputs(torque_out=40.0, n_in=3000.0, floating_sun=True))
assert r3.S_F > r.S_F

## References

- ISO 6336-1:2019 — *Calculation of load capacity of spur and helical gears — Basic principles and influence factors* ($K_v$, $K_{H\beta}$, $K_{H\alpha}$).
- ISO 6336-2:2019 — *Calculation of surface durability (pitting)*.
- ISO 6336-3:2019 — *Calculation of tooth bending strength* (30° tangent, $Y_F$, $Y_S$, $Y_B$, $Y_M$).
- ISO 6336-5:2016 — *Strength and quality of materials*.
- AGMA 6123-B06 — *Design Manual for Enclosed Epicyclic Gear Drives* (planet load sharing $K_\gamma$).